> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 8 · Notebook 02 — Fills, costs and parity

**Sessions:** S3 (Fill & cost models) · S4 (Framework comparison & parity testing) · [Lesson plan](../../docs/lessons/PART_08_BACKTESTING_RISK_PORTFOLIO.md) · graded labs in [`labs/part08/`](../../labs/part08/)

**You will:**
1. Fill limit orders only when the price trades through them.
2. Fill stop orders at the gap when the market opens beyond the stop.
3. Estimate market impact with the square-root law.
4. Prove the engine and the vectorized first look agree, then see what costs do to each strategy.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known truth (known regimes, known Sharpe ratios, pure noise), so every statistic can be checked against reality and every discovery against luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p8lib.py is in notebooks/part08/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p8lib as p

p.use_course_style()

## 1. Limit orders: touching is not filling

A resting limit order joins a queue. If the price only **touches** your level, the orders ahead of you may absorb all the volume. Conservative rule: a **buy** limit fills only if the bar's **low is below** the limit (at `min(open, limit)`, since a gap down fills at the open); a **sell** limit only if the **high is above** it (at `max(open, limit)`). Otherwise `None`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def limit_fill_price(qty, limit, bar):
    if qty > 0:
        return min(bar["open"], limit) if bar["low"] < limit else None
    return max(bar["open"], limit) if bar["high"] > limit else None

bars_ = [{"open": 101.0, "high": 102.0, "low": 99.5, "close": 100.5}, {"open": 101.0, "high": 102.0, "low": 100.0, "close": 100.5},
         {"open": 98.0, "high": 99.0, "low": 97.0, "close": 98.5}, {"open": 99.0, "high": 101.0, "low": 98.0, "close": 100.5},
         {"open": 102.0, "high": 103.0, "low": 101.5, "close": 102.5}]
cases = [(100, 100.0, b) for b in bars_] + [(-100, 101.0, b) for b in bars_]
mine = [p.attempt(limit_fill_price, *cs) for cs in cases]
mine = p.check("limit_fill_price", mine, [p.limit_fill_price(*cs) for cs in cases])
pd.DataFrame({"side": ["buy 100 lmt"] * 5 + ["sell 101 lmt"] * 5, "low": [b["low"] for b in bars_] * 2,
              "high": [b["high"] for b in bars_] * 2, "fill": mine})

Why it matters: every day, bid 1% below yesterday's close and sell at today's close. Require the price to trade through the limit by more and more ticks, and watch the average P&L per fill. The fills you lose are the ones where the price touched your level and bounced: the best ones. That's **adverse selection**, and it's why limit-order strategies look better in backtests than live.

In [ ]:
bars = p.regime_market()
o, h, l, c = (bars[k].to_numpy() for k in ("open", "high", "low", "close"))
rows = []
for k in (0, 1, 3, 10):
    fills = []
    for t in range(1, len(c)):
        lim = round(c[t - 1] * 0.99, 2)
        if round(l[t], 2) <= lim - k * 0.01:
            fills.append(c[t] / min(o[t], lim) - 1)
    rows.append({"require the low this far through (ticks)": k, "fills": len(fills), "mean P&L per fill (bp)": np.mean(fills) * 1e4})
pd.DataFrame(rows).round(1)

## 2. Stop orders and gaps

A stop becomes a market order once triggered. If the market **opens beyond** the stop, you get the open, not your stop: a sell stop triggers if the low reaches it (`low <= stop`) and fills at `min(open, stop)`; a buy stop if `high >= stop` at `max(open, stop)`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def stop_fill_price(qty, stop, bar):
    if qty > 0:
        return max(bar["open"], stop) if bar["high"] >= stop else None
    return min(bar["open"], stop) if bar["low"] <= stop else None

cases = [(-100, 95.0, {"open": 99.0, "high": 100.0, "low": 94.0, "close": 96.0}),    # triggered during the day
         (-100, 95.0, {"open": 90.0, "high": 92.0, "low": 88.0, "close": 91.0}),     # gap down through the stop
         (-100, 95.0, {"open": 99.0, "high": 100.0, "low": 96.0, "close": 97.0}),    # not triggered
         (100, 105.0, {"open": 108.0, "high": 110.0, "low": 107.0, "close": 109.0})] # gap up through a buy stop
mine = [p.attempt(stop_fill_price, *cs) for cs in cases]
mine = p.check("stop_fill_price", mine, [p.stop_fill_price(*cs) for cs in cases])
mine

## 3. Market impact: the square-root law

Big orders move the price. The empirical rule: impact ≈ `k · σ_daily · √(Q / ADV)`, in basis points `1e4 · k · σ · √(|Q| / ADV)`, where `Q` is the order size and `ADV` the average daily volume, in the same units.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def sqrt_impact_bps(order_shares, adv_shares, daily_vol, k=1.0):
    return float(1e4 * k * daily_vol * np.sqrt(abs(order_shares) / adv_shares))

sizes = [1_000, 10_000, 100_000, 1_000_000]
mine = [p.attempt(sqrt_impact_bps, q, 5_000_000, 0.015) for q in sizes]
mine = p.check("sqrt_impact_bps", mine, [p.sqrt_impact_bps(q, 5_000_000, 0.015) for q in sizes])
pd.DataFrame({"order (shares)": sizes, "% of ADV": [q / 5e6 * 100 for q in sizes], "impact (bp)": mine,
              "commission (bp) at $100": [p.ib_fixed_commission(q, 100.0) / (q * 100.0) * 1e4 for q in sizes]}).round(2)

Commissions are a rounding error for size; impact grows with the square root and becomes *the* cost. Capacity (notebook 04) follows from this.

## 4. Parity: two implementations, one answer

The vectorized first look (Part 7) and the event-driven engine must agree when costs are off. A difference means one of them is wrong, and you want to know which before trusting either. Then turn costs on and see who survives.

In [ ]:
sigs = {"SMA 20/100": p.sma_cross_signal(c, 20, 100), "TSMOM 120": p.tsmom_signal(c),
        "daily reversal": np.r_[0.0, np.where(c[1:] < c[:-1], 1.0, 0.0)]}
rows = {}
for name, s in sigs.items():
    fast = p.first_look_pnl(s, o, cost_bps=0)
    eng = p.engine_returns(bars, s)
    rows[name] = {"corr": np.corrcoef(fast, eng)[0, 1], "max |diff| (bp)": np.abs(fast - eng).max() * 1e4,
                  **{f"Sharpe @ {bps} bp": p.sharpe(p.engine_returns(bars, s, slippage_bps=bps)) for bps in (0, 5, 10)}}
pd.DataFrame(rows).T.round(3)

The remaining difference comes from whole shares and from the engine compounding its equity; it is tiny and explained, which is what a parity test is for. Costs barely move the slow strategies; the fast one, already losing before costs, sinks from about −0.5 to −1.3 at 10 bp: a high-turnover idea has to clear a much higher bar.

## Wrap-up

* Limit fills need trade-through; stop fills pay the gap; impact grows like √(size / ADV).
* Parity-test every engine against an independent implementation, then stress costs (2× is part of the robustness scorecard).
* Graded versions: `labs/part08/week25_engine` and Clinic W1 (parity and cost sensitivity).